# Inspect the sample of sources identified in Step 1

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib.ticker as ticker
import numpy as np

import pandas as pd
import pyarrow.parquet as pq

from astrocut import get_center_pixel, ASDFCutout

import os

In [ ]:
in_filter = 'f106' #primary filter
f1 = 'f129' #secondary filter
f2 = 'f158' #tertiary filter
bands = [in_filter, f1, f2]

catalog_name = 'full_catalog_file_locs_'
for i in range(len(bands)):
    if bands[i] == bands[-1]:
        catalog_name = catalog_name + bands[i]
    else:
        catalog_name = catalog_name + bands[i]+'_'
print(catalog_name)

In [ ]:
cat_df = pd.read_parquet('/teams/gz-roman/'+catalog_name+'.parquet')
# make sure there is only one entry per source id. At one point, the 
# above loop reset the src_id counter so this is mostly to test that 
# doesn't happen again. Eventually, we should remove this cell once 
# we are sure that issue does not pop up again.
print(len(cat_df))
print(len(np.unique(cat_df['id_str'])))
# print(np.unique()

# Get the indices of the first occurrences
_, first_occurrence_indices = np.unique(cat_df['id_str'], return_index=True)

# Create a boolean mask of all False values
mask = np.zeros_like(cat_df['id_str'], dtype=bool)

# Mark the first occurrences as True
mask[first_occurrence_indices] = True

# Invert the mask to find all repeated/duplicate indices
repeated_indices = np.where(~mask)[0]

print("Repeated indices:", repeated_indices)
print(len(repeated_indices))
print(len(np.unique(repeated_indices)))

In [ ]:
print(f'Number of sources identified = {len(cat_df)}')
print(cat_df.columns)

### Estimate the size of the region spanned by all identified targets

In [ ]:
# this assumes that the area is roughly a square rotated by 45 degrees with 
# the (min ra, median dec) values at one corner and the (max ra, med dec) values 
# at the opposite corner.
# see here: https://outerspace.stsci.edu/pages/viewpage.action?pageId=389808641&spaceKey=WFISIMWG&title=SOC%2BSimulation%2BProducts
# we should get something like ~ 6deg^2 for the SOC simulations

from astropy.coordinates import SkyCoord
import astropy.units as u

use_band = 'f106'
min_RA = cat_df[f'ra_centroid_{use_band}'].min()
max_RA = cat_df[f'ra_centroid_{use_band}'].max()
med_DEC = cat_df[f'dec_centroid_{use_band}'].median()

c1 = SkyCoord(min_RA, med_DEC, unit='deg')
c2 = SkyCoord(max_RA, med_DEC, unit='deg')
z = c1.separation(c2).deg*u.deg
y = z/np.sqrt(2)
area = y*y
print('Approximate sky area =', area)

### Histograms of magnitudes and sizes

In [ ]:
import matplotlib.pyplot as plt
use_band = 'f158'
sensitivity_5sig_1hour_ptsource = {'f106':27.60, 'f129': 27.60, 'f158': 27.52}
sensitivity_5sig_medtier_ptsource = {'f106': 26.5, 'f129': 26.4, 'f158': 26.4}
PSF_FWHM_per_band = {'f106': 87/(1e3), 'f129': 106/(1e3), 'f158': 128/(1e3)} #arcsec

PSF_area = 1.133*pow(PSF_FWHM_per_band[use_band],2)

pixel_scale = 0.11 #arcsec/pixel

print('PSF area = ', PSF_area, 'arcsec^2', ';', PSF_area/pow(pixel_scale,2), ' pixels')

Cols = ['kron_abmag', 'semimajor']

# vlines = {'kron_abmag':sensitivity_5sig_medtier_ptsource[use_band], 'segment_area': PSF_area}
vlines = {'kron_abmag':sensitivity_5sig_medtier_ptsource[use_band], 'semimajor': 0.11}

# vline_labels = {'kron_abmag': fr'{use_band} 5$\sigma$ pt source sensitivity limit (1 hour)', 'segment_area': f'PSF area = {round(PSF_area,4)}'+ r'arcsec$^2$'}
# vline_labels = {'kron_abmag': fr'{use_band} 5$\sigma$ pt source sensitivity medium tier', 'segment_area': f'PSF area = {round(PSF_area,4)}'+ r'arcsec$^2$'}

vline_labels = {'kron_abmag': fr'{use_band} 5$\sigma$ pt source sensitivity medium tier', 'semimajor': f'pixel scale = {0.11}'+ r'arcsec'}

xlabel_units = {'kron_abmag': '(mags)', 'semimajor': r'(arcsec)'}
# xlabel_units = {'kron_abmag': '(mags)', 'segment_area': r'(pixel$^2$)'}
ext_i = np.where(cat_df[f'is_extended_{use_band}']==True)[0]
no_ext_i = np.where(cat_df[f'is_extended_{use_band}']==False)[0]
print('number of extended sources = ', len(ext_i))
print('number of point sources = ', len(no_ext_i))
for use_col in Cols:
    # use_col = 'kron_abmag'
    bin_min, bin_max = min(cat_df[f'{use_col}_{use_band}']), max(cat_df[f'{use_col}_{use_band}'])
    # print('bin_min, bin_max', bin_min, bin_max)

    if bin_max/bin_min > 50:
        input_data1 = np.log10(cat_df[f'{use_col}_{use_band}'])[ext_i]
        input_data2 = np.log10(cat_df[f'{use_col}_{use_band}'])[no_ext_i]
        
        bin_min1, bin_max1 = min(input_data1), max(input_data1)
        counts1, bins1 = np.histogram(input_data1, bins = np.arange(bin_min1, bin_max1))
        
        bin_min2, bin_max2 = min(input_data2), max(input_data2)
        counts2, bins2 = np.histogram(input_data2, bins = np.arange(bin_min2, bin_max2))
        vlines[use_col] = np.log10(vlines[use_col])
    else:
        input_data1 = cat_df[f'{use_col}_{use_band}'][ext_i]
        counts1, bins1 = np.histogram(input_data1, bins = np.arange(bin_min, bin_max))

        input_data2 = cat_df[f'{use_col}_{use_band}'][no_ext_i]
        counts2, bins2 = np.histogram(input_data2, bins = np.arange(bin_min, bin_max))
    # print(counts, bins)
    
    plt.figure()
    plt.hist(input_data1, bins=20, edgecolor='m', histtype='step', label='extended', fill=False)
    plt.hist(input_data2, bins=20, edgecolor='g', histtype='step', label='point source', fill=False)
    if bin_max/bin_min > 50:
        plt.xlabel(r"log$_{10}$"+f"({use_col} for {use_band}) {xlabel_units[use_col]}")
    else:
        plt.xlabel(f"{use_col} for {use_band} {xlabel_units[use_col]}")
    plt.yscale('log') 
    plt.vlines(vlines[use_col], ymin=0, ymax=1.2*max(max(counts1),max(counts2)), linestyles='--', color='red', label=vline_labels[use_col])
    plt.legend()
    plt.show()


input_data1 = cat_df[f'kron_radius_{use_band}'][ext_i]
input_data2 = cat_df[f'kron_radius_{use_band}'][no_ext_i]
plt.figure()
plt.hist(input_data1, bins=20, edgecolor='m', histtype='step', label='extended', fill=False)
plt.hist(input_data2, bins=20, edgecolor='g', histtype='step', label='point source', fill=False)
plt.xlabel(f"kron_radius for {use_band} (arcsec)")
plt.yscale('log')
plt.vlines(PSF_FWHM_per_band[use_band]/2.0, ymin=0, ymax=1.2*max(max(counts1),max(counts2)), linestyles='--', color='red', label=f'{use_band} PSF FWHM/2 (arcsec) = {PSF_FWHM_per_band[use_band]/2.0}')
plt.vlines(0.11, ymin=0, ymax=1.2*max(max(counts1),max(counts2)), linestyles='--', color='black', label=f'pixel scale (arcsec) = {0.11}')
plt.legend()
plt.show()

input_data1 = cat_df[f'fluxfrac_radius_50_{use_band}'][ext_i]
input_data2 = cat_df[f'fluxfrac_radius_50_{use_band}'][no_ext_i]
plt.figure()
plt.hist(input_data1, bins=20, edgecolor='m', histtype='step', label='extended', fill=False)
plt.hist(input_data2, bins=20, edgecolor='g', histtype='step', label='point source', fill=False)
plt.xlabel(f"kron_radius_50% for {use_band} (arcsec)")
plt.yscale('log')
plt.xscale('log')
plt.vlines(PSF_FWHM_per_band[use_band]/2.0, ymin=0, ymax=1.2*max(max(counts1),max(counts2)), linestyles='--', color='red', label=f'{use_band} PSF FWHM/2 (arcsec) = {PSF_FWHM_per_band[use_band]/2.0}')
plt.vlines(0.11, ymin=0, ymax=1.2*max(max(counts1),max(counts2)), linestyles='--', color='black', label=f'pixel scale (arcsec) = {0.11}')
plt.legend()
plt.show()

### I'm curious if the effective radius is always larger than the kron radius. Plot their ratio below:

In [ ]:
ext_kron_over_pix_i = np.where((cat_df[f'kron_radius_{use_band}']>0.11) & (cat_df[f'is_extended_{use_band}']==True))[0]
noext_kron_over_pix_i = np.where((cat_df[f'kron_radius_{use_band}']>0.11) & (cat_df[f'is_extended_{use_band}']==False))[0]

ext_kron50_over_pix_i = np.where((cat_df[f'fluxfrac_radius_50_{use_band}']>0.11) & (cat_df[f'is_extended_{use_band}']==True))[0]
noext_kron50_over_pix_i = np.where((cat_df[f'fluxfrac_radius_50_{use_band}']>0.11) & (cat_df[f'is_extended_{use_band}']==False))[0]


In [ ]:
input_data1 = cat_df[f'fluxfrac_radius_50_{use_band}']/cat_df[f'kron_radius_{use_band}'][ext_kron_over_pix_i]
input_data2 = cat_df[f'fluxfrac_radius_50_{use_band}']/cat_df[f'kron_radius_{use_band}'][noext_kron_over_pix_i]
plt.figure()
plt.title('Kron radius larger than pixel scale')
plt.hist(input_data1, bins=20, edgecolor='m', histtype='step', label='extended', fill=False)
plt.hist(input_data2, bins=20, edgecolor='g', histtype='step', label='point source', fill=False)
plt.xlabel(f"radius_fluxfrac_50%/kron radius for {use_band}")
plt.yscale('log')
plt.xscale('log')
# plt.vlines(PSF_FWHM_per_band[use_band]/2.0, ymin=0, ymax=1.2*max(max(counts1),max(counts2)), linestyles='--', color='red', label=f'{use_band} PSF FWHM/2 (arcsec) = {PSF_FWHM_per_band[use_band]/2.0}')
# plt.vlines(0.11, ymin=0, ymax=1.2*max(max(counts1),max(counts2)), linestyles='--', color='black', label=f'pixel scale (arcsec) = {0.11}')
plt.legend()
plt.show()

### Sizes vs magnitude: 

In [ ]:
Cols = ['fluxfrac_radius_50', 'segment_area']

vlines = {'fluxfrac_radius_50': 0.11, 'segment_area': 0.11*0.11}

vline_labels = {'fluxfrac_radius_50': f'pixel scale = {0.11} (arcsec)', 'segment_area': f'pixel area = {round(0.11*0.11,4)}'+ r'arcsec$^2$'}

xlabel_units = {'fluxfrac_radius_50': '(arcsec)', 'segment_area': r'(arcsec$^2$)'}


for use_col in Cols:
    x1, y1 = cat_df[f'{use_col}_{use_band}'][ext_i], cat_df[f'kron_abmag_{use_band}'][ext_i]
    x2, y2 = cat_df[f'{use_col}_{use_band}'][no_ext_i], cat_df[f'kron_abmag_{use_band}'][no_ext_i]
    plt.figure()
    plt.scatter(x2, y2, s=5, c='g', alpha=0.25, label='point source')
    plt.scatter(x1, y1, s=5, c='m', alpha=0.25, label='extended')

    plt.xlabel(f'{use_col}_{use_band} {xlabel_units[use_col]}')
    plt.ylabel(f'kron_abmag_{use_band}')
    plt.vlines(vlines[use_col], ymin=min(min(y1),min(y2)), ymax=max(max(y1),max(y2)), linestyle='--', color='red', label = vline_labels[use_col])
    plt.legend()
    plt.xscale('log')
    plt.show()

### Sizes vs magnitude: with restirction on size 

In [ ]:
# use_band = 'f106'
Cols = ['fluxfrac_radius_50', 'segment_area']

vlines = {'fluxfrac_radius_50': 0.11, 'segment_area': 0.11*0.11}

vline_labels = {'fluxfrac_radius_50': f'pixel scale = {0.11} (arcsec)', 'segment_area': f'pixel area = {round(0.11*0.11,4)}'+ r'arcsec$^2$'}

xlabel_units = {'fluxfrac_radius_50': '(arcsec)', 'segment_area': r'(arcsec$^2$)'}


for use_col in Cols:
    x1, y1 = cat_df[f'{use_col}_{use_band}'][ext_kron_over_pix_i], cat_df[f'kron_abmag_{use_band}'][ext_kron_over_pix_i]
    x2, y2 = cat_df[f'{use_col}_{use_band}'][noext_kron_over_pix_i], cat_df[f'kron_abmag_{use_band}'][noext_kron_over_pix_i]
    plt.figure()
    plt.title('Kron radius > pixel scale (0.11 arcsec)')
    plt.scatter(x2, y2, s=5, alpha=0.25, c='g', label='point source')
    plt.scatter(x1, y1, s=5, alpha=0.25, c='m', label='extended')
    
    plt.xlabel(f'{use_col}_{use_band} {xlabel_units[use_col]}')
    plt.ylabel(f'kron_abmag_{use_band}')
    plt.vlines(vlines[use_col], ymin=min(min(y1),min(y2)), ymax=max(max(y1),max(y2)), linestyle='--', color='red', label = vline_labels[use_col])
    plt.legend()
    plt.xscale('log')
    plt.show()

In [ ]:
# use_band = 'f106'
Cols = ['fluxfrac_radius_50', 'segment_area']

vlines = {'fluxfrac_radius_50': 0.11, 'segment_area': 0.11*0.11}

vline_labels = {'fluxfrac_radius_50': f'pixel scale = {0.11} (arcsec)', 'segment_area': f'pixel area = {round(0.11*0.11,4)}'+ r'arcsec$^2$'}

xlabel_units = {'fluxfrac_radius_50': '(arcsec)', 'segment_area': r'(arcsec$^2$)'}


for use_col in Cols:
    x1, y1 = cat_df[f'{use_col}_{use_band}'][ext_kron50_over_pix_i], cat_df[f'kron_abmag_{use_band}'][ext_kron50_over_pix_i]
    x2, y2 = cat_df[f'{use_col}_{use_band}'][noext_kron50_over_pix_i], cat_df[f'kron_abmag_{use_band}'][noext_kron50_over_pix_i]
    plt.figure()
    plt.title('Kron fluxfrac_50 radius > pixel scale (0.11 arcsec)')
    plt.scatter(x2, y2, s=5, alpha=0.25, c='g', label='point source')
    plt.scatter(x1, y1, s=5, alpha=0.25, c='m', label='extended')
    
    plt.xlabel(f'{use_col}_{use_band} {xlabel_units[use_col]}')
    plt.ylabel(f'kron_abmag_{use_band}')
    plt.vlines(vlines[use_col], ymin=min(min(y1),min(y2)), ymax=max(max(y1),max(y2)), linestyle='--', color='red', label = vline_labels[use_col])
    plt.legend()
    plt.xscale('log')
    plt.show()

### Color vs magnitude

In [ ]:
use_bands = ['f106', 'f158']
f1_ext = cat_df[f'kron_abmag_{use_bands[0]}'][ext_i]
f2_ext = cat_df[f'kron_abmag_{use_bands[1]}'][ext_i]

f1_noext = cat_df[f'kron_abmag_{use_bands[0]}'][no_ext_i]
f2_noext = cat_df[f'kron_abmag_{use_bands[1]}'][no_ext_i]

plt.figure()
plt.scatter(f2_ext-f1_ext, f1_ext, s=10, c='m', alpha=0.5, label='extended')
plt.scatter(f2_noext-f1_noext, f1_noext, s=10, c='g', alpha=0.5, label='point source')
plt.xlabel(f'kron_abmag_{use_bands[1]} - kron_abmag_{use_bands[0]}')
plt.ylabel(f'kron_abmag_{use_bands[0]}')
plt.legend()
plt.show()


### Redo color magnitude without source with kron radius smaller than pixel scale

In [ ]:
over_pixscale_i = np.where(cat_df[f'kron_radius_{use_band}'] > 0.11)
# over_pixscale_i = np.where(cat_df[f'fluxfrac_radius_50_{use_band}'] > 0.11)

use_bands = ['f106', 'f158']
f1 = cat_df[f'kron_abmag_{use_bands[0]}'].iloc[over_pixscale_i]
f2 = cat_df[f'kron_abmag_{use_bands[1]}'].iloc[over_pixscale_i]

plt.figure()
plt.scatter(f2-f1, f1, s=10, c='green', alpha=0.1)
plt.xlabel(f'kron_abmag_{use_bands[1]} - kron_abmag_{use_bands[0]}')
plt.ylabel(f'kron_abmag_{use_bands[0]}')
plt.show()

### Display cutouts for source with various conditions to see how they visually appear

Want to determine which flags/conditions are useful for selecting sources that are extended enough to be classified in galaxy zoo. 

### Display sources with:
    - kron radius > pixel scale (0.11 arcsec), 
    - fluxfrac_50 > pixel scale, and 
    - both true
    - fluxfrac_50/kron radius > 1.0 and both larger than pixel scale
    - kron radius > pixel scale + is_extended == True
    - kron radius > pixel scale + is_extended == False

In [ ]:
# select sources:
# is_extended == True
extended_cat = cat_df.iloc[ext_i]

# is_extended==False
pointsource_cat = cat_df.iloc[no_ext_i]

# kron radius over pixel scale
kron_over_pix_i = np.where((cat_df[f'kron_radius_{use_band}']>0.11))[0]
kron_over_pix = cat_df.iloc[kron_over_pix_i]

# fluxfrac_50 over pixel scale
fluxfrac_50_rad_over_pix_i = np.where((cat_df[f'fluxfrac_radius_50_{use_band}']>0.11))[0]
fluxfrac_50_rad_over_pix = cat_df.iloc[fluxfrac_50_rad_over_pix_i]

# both kron and fluxfrac_50_rad over pixel scale
all_rad_over_pix_scale_i = np.where((cat_df[f'kron_radius_{use_band}']>0.11) & (cat_df[f'fluxfrac_radius_50_{use_band}']>0.11))[0]
all_rad_over_pix_scale = cat_df.iloc[all_rad_over_pix_scale_i]

# both kron and fluxfrac_50_rad over pixel scale, and fluxfrac_50_rad > kron radius
rad_over_pix_low_concentration_i = np.where((cat_df[f'kron_radius_{use_band}']>0.11) & (cat_df[f'fluxfrac_radius_50_{use_band}']>0.11) & (cat_df[f'fluxfrac_radius_50_{use_band}']/cat_df[f'kron_radius_{use_band}'] > 1.0))[0]
rad_over_pix_low_concentration = cat_df.iloc[rad_over_pix_low_concentration_i]

kron_over_pix_ext_i = np.where((cat_df[f'kron_radius_{use_band}']>0.11) & (cat_df[f'is_extended_{use_band}']==True))
kron_over_pix_ext = cat_df.iloc[kron_over_pix_ext_i]

kron_over_pix_noext_i = np.where((cat_df[f'kron_radius_{use_band}']>0.11) & (cat_df[f'is_extended_{use_band}']==False))
kron_over_pix_noext = cat_df.iloc[kron_over_pix_noext_i]

SELECTIONS = [extended_cat, pointsource_cat, kron_over_pix, fluxfrac_50_rad_over_pix, all_rad_over_pix_scale, rad_over_pix_low_concentration, kron_over_pix_ext, kron_over_pix_noext]

TITLES = ['Is Extended', 'Is point source', 'Kron radius > pixel scale', 'fluxfrac_50_radius >  pixel scale', 'All radii over pixel scale', 'All radii over pixel scale + low concentration', 'Kron over pixel scale + is_extended==True', 'Kron over pixel scale + is_extended==False']
SAVENAMES = ['is_extendedTrue', 'is_extendedFalse', 'kron_rad_over_pixscale', 'fluxfrac_50_rad_over_pixscale', 'all_radii_over_pixscale', 'all_radii_over_pixscale_low_concentration', 'kron_over_pixscale_extendedTrue', 'kron_over_pixscale_extendedFalse']

In [ ]:
num = 25
cutout_size = [64,64] #pixels

for i in range(len(SELECTIONS)-2, len(SELECTIONS)):
    print(f'Making selection for {TITLES[i]}')
    random_ids = np.random.randint(0,len(SELECTIONS[i]), num)
    ra, dec = SELECTIONS[i]['ra_centroid_f106'].values[random_ids],SELECTIONS[i]['dec_centroid_f106'].values[random_ids]
    
    center_coord = SkyCoord(ra, dec, unit='deg')
    # print(center_coord)
    
    file_names = SELECTIONS[i]['asdf_loc_f106'].iloc[random_ids]
    
    # Create the image cutout
    IMS = []
    for j in range(len(random_ids)):
        asdf_file_uri = file_names.iloc[j]
        asdf_cutout = ASDFCutout(asdf_file_uri, center_coord[j], cutout_size, fill_value=0)
        IMS.append(asdf_cutout.cutouts)

    fig, axs = plt.subplots(figsize=(5,5), sharex=True, sharey=True, ncols=5, nrows=5)
    ax = axs.ravel()
    plt.suptitle(TITLES[i])
    for j in range(len(IMS)):
        ax[j].imshow(IMS[j][0].data, norm=LogNorm(vmin=np.nanmin(np.clip(IMS[j][0].data, a_min=1e-2, a_max=np.nanmax(IMS[j][0].data))), vmax=np.nanmax(IMS[j][0].data)))
    ax[j].set_xticks([])
    ax[j].set_yticks([])

    output_file = f'/teams/gz-roman/Sample_Analysis_Figures/{SAVENAMES[i]}.png'
    if not os.path.isfile(output_file):
        plt.savefig(output_file, bbox_inches='tight', transparent=True)
        plt.close(fig)
    
    
    plt.show()

In [ ]:
# def show_grid(df, col, nrows=8, ncols=10, scale=1.4):
    '''
    show example images from that represent quantiles of the specified column. 
    
    '''
    
#     df = df.copy()
    
    
#     # divide into percentiles
#     quantile_intervals = pd.qcut(df[col], nrows)
#     # print(quantile_intervals)
#     df['interval'] = quantile_intervals
#     unique_intervals = pd.unique(quantile_intervals)  # will use, but need to sort
    
#     midpoint_intervals = np.array([x.mid for x in unique_intervals])
#     midpoint_interval_order = np.argsort(midpoint_intervals)
#     unique_intervals_sorted = unique_intervals[midpoint_interval_order]
    
#     fig, rows = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols*scale, nrows*scale))
    
#     for row_n, quantile in enumerate(unique_intervals_sorted):
#         qdf = df[df['interval'] == quantile].sample(ncols, random_state=42).reset_index()
#         row = rows[row_n]
        
#         for ax_n, ax in enumerate(row):
#             galaxy = qdf.iloc[ax_n]
#             ax.imshow(plt.imread(galaxy['mtf_loc']))
#             # ax.axis('off')
#             if ax_n == 0:
#                 ax.set_ylabel(quantile.mid)
#                 if row_n == 0:
#                     ax.set_title(col)
#             # ax.xaxis.set_visible(False)
#             # ax.yaxis.set_visible(False)
#             ax.xaxis.set_major_locator(ticker.NullLocator())
#             ax.xaxis.set_minor_locator(ticker.NullLocator())
#             ax.yaxis.set_major_locator(ticker.NullLocator())
#             ax.yaxis.set_minor_locator(ticker.NullLocator())
        
    
# show_grid(df, 'AREA')

In [ ]:
# def show_grid(df, col, nrows=8, ncols=10, scale=1.4):

nrows=8 
ncols=10 
scale=1.4
col = f'kron_abmag_{use_band}'
cutout_size = [64,64] #pixels

df = cat_df.copy()


# divide into percentiles
quantile_intervals = pd.qcut(df[col], nrows)
# print(quantile_intervals)

df['interval'] = quantile_intervals
unique_intervals = pd.unique(quantile_intervals).dropna()  # will use, but need to sort
# print(unique_intervals)

midpoint_intervals = np.array([x.mid for x in unique_intervals])
print(midpoint_intervals)
midpoint_interval_order = np.argsort(midpoint_intervals)
# print(midpoint_intervals[midpoint_interval_order])

unique_intervals_sorted = unique_intervals[midpoint_interval_order]

print(unique_intervals_sorted)

fig, rows = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols*scale, nrows*scale))
# print(rows)
for row_n, quantile in enumerate(unique_intervals_sorted):

    success = False

    while not success:
    
        try:
            qdf = df[df['interval'] == quantile].sample(ncols).reset_index() #select ncols galaxies that match the quantile range
            # print(qdf)
            row = rows[row_n]
            
            for ax_n, ax in enumerate(row): #loop through each column of the row
                galaxy = qdf.iloc[ax_n]
                # print(galaxy)
                galaxy_image = ASDFCutout(galaxy['asdf_loc_f106'], SkyCoord(galaxy[f'ra_centroid_{use_band}'], galaxy[f'dec_centroid_{use_band}'], unit='deg'), cutout_size, fill_value=0)
                ax.imshow(galaxy_image.cutouts[0].data)
                # ax.axis('off')
                if ax_n == 0:
                    ax.set_ylabel(round(quantile.mid, 3))
                    if row_n == 0:
                        ax.set_title(col)
                # ax.xaxis.set_visible(False)
                # ax.yaxis.set_visible(False)
                ax.xaxis.set_major_locator(ticker.NullLocator())
                ax.xaxis.set_minor_locator(ticker.NullLocator())
                ax.yaxis.set_major_locator(ticker.NullLocator())
                ax.yaxis.set_minor_locator(ticker.NullLocator())
                success = True
                
        except Exception as e:
            # 1. Print the basic error message
            print(f"Row {row_n} failed: {e}")
            print("Trying another set of sources...")
        